## Imports

In [191]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor, Pool

import optuna
import pickle

In [215]:
data = pd.read_csv("data/somon_ml_clear_amir3.csv")

In [167]:
data.head()

,price,rooms,area_m2,floor,district,build_type,renovation,bathroom,heating,condition,techpassport
0,105000,1,32,2,Другое,Вторичный рынок,Новый ремонт,Раздельный,Нет,Построено,Есть
1,110000,2,40,3,Другое,Вторичный рынок,Без ремонта (коробка),Совмещенный,Нет,Построено,Есть
2,115000,3,9,1,Другое,Новостройка,Без ремонта (коробка),Раздельный,Нет,На стадии строительства,Есть
3,120000,2,6,1,И. Сомони,Новостройка,Без ремонта (коробка),Раздельный,Нет,Построено,Есть
4,125000,1,25,8,Другое,Новостройка,Без ремонта (коробка),Совмещенный,Нет,На стадии строительства,Нет


In [168]:
data.describe(include='number')

,price,rooms,area_m2,floor
count,1.007000e+04,10070.000000,10070.000000,10070.000000
mean,7.817983e+05,2.329494,82.107349,7.701589
std,5.078719e+05,0.880638,84.806666,4.553325
min,1.050000e+05,1.000000,0.000000,1.000000
25%,4.250000e+05,2.000000,57.000000,4.000000
50%,6.850000e+05,2.000000,72.000000,7.000000
75%,9.800000e+05,3.000000,95.000000,11.000000
max,6.500000e+06,6.000000,4438.000000,22.000000


In [216]:
CATEGORICAL_FEATURES = ['district', 'build_type', 'renovation', 'bathroom', 'heating', 'condition', 'techpassport']
for i in CATEGORICAL_FEATURES:
    print(i, data[i].nunique())

district 5
build_type 2
renovation 3
bathroom 2
heating 2
condition 2
techpassport 2


In [199]:
data.shape

(9530, 11)

### Preprocessing

In [217]:
IQR = data['price'].quantile(0.75) - data['price'].quantile(0.25)
upper_bound = data['price'].quantile(0.75) + 1.5 * IQR
data = data[data['price']<= upper_bound]

In [218]:
IQR = data['area_m2'].quantile(0.75) - data['area_m2'].quantile(0.25)
upper_bound = data['area_m2'].quantile(0.75) + 1.5 * IQR
data = data[(data['area_m2']<= upper_bound) & (data['area_m2']> 18)]

In [219]:
X = data.drop('price', axis=1)
y = data['price']

In [222]:
data.shape

(9507, 11)

In [249]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42
)

In [250]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5704, 10), (3803, 10), (5704,), (3803,))

### Model Training

In [251]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=CATEGORICAL_FEATURES
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=CATEGORICAL_FEATURES
)

In [134]:
def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 600, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 50.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "border_count": trial.suggest_int("border_count", 32, 255),

        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "verbose": False
    }

    model = CatBoostRegressor(**params)

    model.fit(
        train_pool,
        eval_set=[test_pool],
        early_stopping_rounds=100,
        use_best_model=True
    )

    preds = model.predict(test_pool)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return rmse


In [187]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=40,
    show_progress_bar=True
)


[I 2026-01-04 13:51:11,407] A new study created in memory with name: no-name-d9b4f240-7890-4027-a095-f3265fed039a


  0%|          | 0/40 [00:00<?, ?it/s]

[W 2026-01-04 13:51:17,381] Trial 0 failed with parameters: {'iterations': 1124, 'learning_rate': 0.17254716573280354, 'depth': 9, 'l2_leaf_reg': 0.6502468545951017, 'bagging_temperature': 0.15601864044243652, 'random_strength': 0.3119890406724053, 'border_count': 45} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "C:\Users\School\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\School\AppData\Local\Temp\ipykernel_13260\2885438550.py", line 20, in objective
    model.fit(
  File "C:\Users\School\AppData\Local\Programs\Python\Python312\Lib\site-packages\catboost\core.py", line 5873, in fit
    return self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [161]:
best_params = study.best_params

final_model = CatBoostRegressor(
    **best_params,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

0:	learn: 353932.2418024	test: 352039.0646020	best: 352039.0646020 (0)	total: 54.7ms	remaining: 1m 43s
100:	learn: 133385.7587755	test: 142628.3886925	best: 142628.3886925 (100)	total: 5.93s	remaining: 1m 45s
200:	learn: 122111.3660197	test: 136549.4471888	best: 136549.4471888 (200)	total: 11s	remaining: 1m 32s
300:	learn: 115658.5442247	test: 134377.5404845	best: 134377.5404845 (300)	total: 16.2s	remaining: 1m 25s
400:	learn: 110592.1430708	test: 133119.0606322	best: 133119.0606322 (400)	total: 21.3s	remaining: 1m 19s
500:	learn: 106268.3760907	test: 132052.7484447	best: 132052.7484447 (500)	total: 26.3s	remaining: 1m 12s
600:	learn: 102066.4747770	test: 131329.7681980	best: 131329.7681980 (600)	total: 31.2s	remaining: 1m 6s
700:	learn: 98492.2494454	test: 130645.3693150	best: 130645.3693150 (700)	total: 36.4s	remaining: 1m 1s
800:	learn: 95525.9000183	test: 130193.3105173	best: 130193.3105173 (800)	total: 41.3s	remaining: 56.1s
900:	learn: 92611.6484560	test: 129818.1020575	best: 129

In [ ]:
final_model.fit(
    train_pool,
    eval_set=[test_pool],
    early_stopping_rounds=100,
    use_best_model=True
)

0:	learn: 343456.5481309	test: 341982.2276748	best: 341982.2276748 (0)	total: 54.5ms	remaining: 1m 42s
100:	learn: 134082.9204956	test: 132151.4386745	best: 132151.4386745 (100)	total: 8.03s	remaining: 2m 22s
200:	learn: 124102.3562937	test: 123082.3011634	best: 123082.3011634 (200)	total: 14.4s	remaining: 2m 1s
300:	learn: 118505.6416238	test: 118526.4072771	best: 118526.4072771 (300)	total: 20.5s	remaining: 1m 47s
400:	learn: 113946.6197903	test: 114847.2101562	best: 114847.2101562 (400)	total: 32.6s	remaining: 2m 1s
500:	learn: 109790.8402829	test: 111555.9775231	best: 111555.9775231 (500)	total: 39.3s	remaining: 1m 48s


In [253]:
y_pred = final_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:,.0f} TJS")
print(f"MAE:  {mae:,.0f} TJS")
print(f"MAPE:  {mape*100:,.0f}%")
print(f"R²:   {r2:.3f}")


RMSE: 134,634 TJS
MAE:  94,876 TJS
MAPE:  16%
R²:   0.852


In [165]:
feature_importance = final_model.get_feature_importance(train_pool)
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance
}).sort_values(by="importance", ascending=False)

print(importance_df.head(10))


        feature  importance
1       area_m2   31.061458
5    renovation   18.141687
7       heating   12.923848
3      district   12.326089
2         floor   10.425563
0         rooms    4.980860
8     condition    3.714613
9  techpassport    3.313695
4    build_type    1.618393
6      bathroom    1.493795


In [163]:
with open("catboost_rmse128_r877.pkl", 'wb') as file:
    pickle.dump(final_model, file)

### Random Forest

In [231]:
ohe = OneHotEncoder(sparse_output=False)

encoded = ohe.fit_transform(X_train[CATEGORICAL_FEATURES])
one_hot_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(CATEGORICAL_FEATURES), index=X_train.index)
train_encoded = pd.concat([X_train.drop(CATEGORICAL_FEATURES, axis=1), one_hot_df], axis=1)

encoded = ohe.transform(X_test[CATEGORICAL_FEATURES])
one_hot_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(CATEGORICAL_FEATURES), index=X_test.index)
test_encoded = pd.concat([X_test.drop(CATEGORICAL_FEATURES, axis=1), one_hot_df], axis=1)

In [241]:
rf = RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=5)

rf.fit(train_encoded, y_train)

preds = rf.predict(test_encoded)
rmse = np.sqrt(mean_squared_error(y_test, preds))

In [243]:
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
mape = mean_absolute_percentage_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"RMSE: {rmse:,.0f} TJS")
print(f"MAE:  {mae:,.0f} TJS")
print(f"MAPE:  {mape*100:,.0f}%")
print(f"R²:   {r2:.3f}")

RMSE: 131,806 TJS
MAE:  93,375 TJS
MAPE:  16%
R²:   0.856


In [237]:
with open("ohe_rf_rmse129_r861.pkl", 'wb') as file:
    pickle.dump(ohe, file)